# MandiLens India — Environment Setup

**Notebook:** `00_environment_setup.ipynb`  
**Phase:** 2 — Environment and Data Acquisition  
**Purpose:** Validate the local Python environment, connect the notebook to the existing MandiLens package, verify configuration and storage conventions, and run small, in-memory dependency smoke tests.

> **Data safety:** This notebook does not download, scan, or load the 76-million-record dataset. It does not modify raw data. Dataset acquisition and schema verification belong to later, dedicated steps.

## 1. Learning objectives

By the end of this notebook, you should be able to:

1. Confirm that the active Python interpreter is supported by the repository.
2. Discover the repository root reliably, including when Jupyter starts in a subdirectory.
3. Import the existing `mandilens` package and reuse its centralized path utilities.
4. Validate `config/project_config.yaml` through the project's configuration manager.
5. Initialize the project's configured logger without duplicating logging configuration.
6. Audit installed dependencies and identify missing packages from `requirements.txt`.
7. Test Pandas, Polars, PyArrow and DuckDB using tiny in-memory examples only.
8. Save a reproducible environment report under `artifacts/metadata/`.

### Execution policy

- Use **Python 3.12 or 3.13**. The repository's `pyproject.toml` declares `>=3.12,<3.14`.
- Run the notebook from the repository, preferably after installing the project in an isolated virtual environment.
- Execute cells from top to bottom. Restart the kernel after changing Python environments or package installations.
- Never paste API tokens, passwords or `.env` values into notebook cells or outputs.

## 2. Prepare the Python environment

The repository's `requirements.txt` is the authoritative dependency list during early development. A clean virtual environment is strongly recommended.

### Windows PowerShell

```powershell
# From the MandiLens repository root
py -3.12 -m venv .venv
.\.venv\Scripts\Activate.ps1
python -m pip install --upgrade pip
python -m pip install -r requirements.txt
python -m pip install -e .
python -m ipykernel install --user --name mandilens-india --display-name "Python (MandiLens India)"
jupyter lab
```

### macOS / Linux

```bash
# From the MandiLens repository root
python3.12 -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip
python -m pip install -r requirements.txt
python -m pip install -e .
python -m ipykernel install --user --name mandilens-india --display-name "Python (MandiLens India)"
jupyter lab
```

In Jupyter, select the **Python (MandiLens India)** kernel. If your environment already exists, do not recreate it; activate it and run the relevant installation or verification commands. Python 3.14 is outside the project's declared interpreter range.

## 3. Check the active Python interpreter

This is a hard compatibility check. Resolve an unsupported interpreter before continuing; do not work around the repository metadata by editing `pyproject.toml`.

In [1]:
from __future__ import annotations

import platform
import sys
from importlib import metadata
from pathlib import Path

PYTHON_VERSION = sys.version_info[:3]
PYTHON_SUPPORTED = (3, 12) <= PYTHON_VERSION[:2] < (3, 14)

print(f"Python version : {platform.python_version()}")
print(f"Implementation : {platform.python_implementation()}")
print(f"Operating system: {platform.system()} ({platform.machine()})")
print(f"Interpreter check: {'PASS' if PYTHON_SUPPORTED else 'FAIL'}")

if not PYTHON_SUPPORTED:
    raise RuntimeError(
        "Unsupported Python interpreter. MandiLens requires Python >=3.12,<3.14. "
        "Create or activate a Python 3.12/3.13 environment, install the project dependencies, "
        "select that environment as the Jupyter kernel, and restart the notebook."
    )

Python version : 3.12.10
Implementation : CPython
Operating system: Windows (AMD64)
Interpreter check: PASS


## 4. Discover the repository root

The root is identified by three repository markers rather than by assuming that the notebook kernel starts in a particular directory. `MANDILENS_PROJECT_ROOT`, when set, takes precedence and must point to a valid repository root.

In [2]:
import os
import sys
from pathlib import Path

PROJECT_MARKERS = (
    Path("pyproject.toml"),
    Path("config/project_config.yaml"),
    Path("src/mandilens/__init__.py"),
)


def is_mandilens_root(candidate: Path) -> bool:
    """Return True when candidate contains the required project markers."""
    return all((candidate / marker).is_file() for marker in PROJECT_MARKERS)


configured_root = os.environ.get("MANDILENS_PROJECT_ROOT", "").strip()
if configured_root:
    candidate_root = Path(configured_root).expanduser().resolve()
    if not is_mandilens_root(candidate_root):
        raise FileNotFoundError(
            "MANDILENS_PROJECT_ROOT is set, but it does not contain the required "
            "MandiLens project markers. Correct the variable and restart the kernel."
        )
    PROJECT_ROOT = candidate_root
else:
    PROJECT_ROOT = None
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if is_mandilens_root(candidate):
            PROJECT_ROOT = candidate.resolve()
            break

    if PROJECT_ROOT is None:
        raise FileNotFoundError(
            "Could not discover the MandiLens repository from the notebook's current "
            "working directory. Start Jupyter from the repository, or set the "
            "MANDILENS_PROJECT_ROOT environment variable to the repository root."
        )

# Set this before importing mandilens: PATHS is initialized when its module loads.
os.environ["MANDILENS_PROJECT_ROOT"] = str(PROJECT_ROOT)
SOURCE_ROOT = PROJECT_ROOT / "src"
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))

print("Repository root: detected")
print("Required project markers: PASS")
print("Source package directory: available" if SOURCE_ROOT.is_dir() else "Source package directory: MISSING")

Repository root: detected
Required project markers: PASS
Source package directory: available


## 5. Import the existing MandiLens foundation

This cell reuses the repository's `ProjectPaths` registry. It does not create directories merely by importing the module.

In [3]:
from mandilens import __all__ as mandilens_public_api
from mandilens.utils.paths import PATHS, ProjectPaths

if PATHS.root != PROJECT_ROOT:
    raise RuntimeError(
        "The imported MandiLens PATHS registry points to a different project root. "
        "Restart the kernel after setting MANDILENS_PROJECT_ROOT."
    )

# Validate that the path manager resolves the same root independently.
verified_paths = ProjectPaths.discover(PROJECT_ROOT)
assert verified_paths.root == PROJECT_ROOT

print("MandiLens package import: PASS")
print("Centralized path registry: PASS")
print("Project root consistency: PASS")

MandiLens package import: PASS
Centralized path registry: PASS
Project root consistency: PASS


## 6. Initialize the standard working directories

Only directories needed for the early data workflow are created. The original source files are never edited by this setup step.

In [4]:
DIRECTORIES_TO_INITIALIZE = (
    PATHS.notebooks_dir,
    PATHS.raw_data_dir,
    PATHS.interim_data_dir,
    PATHS.processed_data_dir,
    PATHS.external_data_dir,
    PATHS.warehouse_dir,
    PATHS.artifacts_dir,
    PATHS.artifacts_dir / "metadata",
    PATHS.images_dir,
    PATHS.images_dir / "environment_setup",
    PATHS.logs_dir,
)

created_or_existing = PATHS.ensure_directories(*DIRECTORIES_TO_INITIALIZE)

# Confirm that every selected directory is inside the repository and available.
assert all(path.is_dir() for path in created_or_existing)
assert all(path.is_relative_to(PROJECT_ROOT) for path in created_or_existing)

print(f"Directory initialization: PASS ({len(created_or_existing)} directories checked)")
print("Raw-data directory policy: source files remain untouched")

Directory initialization: PASS (11 directories checked)
Raw-data directory policy: source files remain untouched


## 7. Load local environment variables safely

A `.env` file is optional at this stage. It is read only when present; its contents and secret values are never displayed. The `.env.example` template is a separate foundation item and is not a blocker for local setup.

In [5]:
from dotenv import load_dotenv

ENV_FILE = PROJECT_ROOT / ".env"
ENV_TEMPLATE_FILE = PROJECT_ROOT / ".env.example"

if ENV_FILE.is_file():
    load_dotenv(dotenv_path=ENV_FILE, override=False)
    ENV_FILE_STATUS = "loaded (values not displayed)"
else:
    ENV_FILE_STATUS = "not present (optional for this notebook)"

ENV_TEMPLATE_STATUS = "present" if ENV_TEMPLATE_FILE.is_file() else "not present yet"

print(f"Local .env file: {ENV_FILE_STATUS}")
print(f".env.example template: {ENV_TEMPLATE_STATUS}")
print("Credential safety check: no environment values were printed")

Local .env file: not present (optional for this notebook)
.env.example template: present
Credential safety check: no environment values were printed


## 8. Validate the central project configuration

The project configuration loader checks YAML parsing and the required top-level structure. Dataset figures shown here are publisher-reported configuration values, not a verified local row count or schema audit.

In [6]:
from mandilens.utils.config import ConfigurationError, load_config

try:
    PROJECT_CONFIG = load_config("project")
except ConfigurationError:
    raise

required_config_sections = ("project", "environment", "paths")
missing_config_sections = [
    section for section in required_config_sections
    if section not in PROJECT_CONFIG
]
if missing_config_sections:
    raise ValueError(
        "Project configuration is missing required sections: "
        + ", ".join(missing_config_sections)
    )

PROJECT_NAME = PROJECT_CONFIG["project"].get("name", "MandiLens India")
PROJECT_VERSION = PROJECT_CONFIG["project"].get("version", "unknown")
DATASET_PROFILE = PROJECT_CONFIG.get("dataset", {}).get("publisher_reported_profile", {})

print(f"Configuration validation: PASS")
print(f"Project: {PROJECT_NAME} ({PROJECT_VERSION})")
print("Dataset profile (publisher-reported; not yet locally verified):")
print(f"  Approximate records: {DATASET_PROFILE.get('approximate_record_count', 'not specified'):,}" if isinstance(DATASET_PROFILE.get("approximate_record_count"), int) else f"  Approximate records: {DATASET_PROFILE.get('approximate_record_count', 'not specified')}")
print(f"  Reported yearly Parquet files: {DATASET_PROFILE.get('reported_yearly_parquet_files', 'not specified')}")
print(f"  Reported format: {DATASET_PROFILE.get('reported_format', 'not specified')}")
print(f"  Schema audit status: {PROJECT_CONFIG.get('dataset', {}).get('schema_audit', {}).get('schema_verified', 'not specified')}")

if PROJECT_CONFIG.get("dataset", {}).get("complete_dataset_required") is not True:
    raise ValueError("Project configuration must preserve the complete nationwide dataset scope.")

Configuration validation: PASS
Project: MandiLens India (0.1.0)
Dataset profile (publisher-reported; not yet locally verified):
  Approximate records: 75,960,000
  Reported yearly Parquet files: 26
  Reported format: parquet
  Schema audit status: False


## 9. Initialize centralized logging

The logger reads its settings from `config/project_config.yaml`. It uses the `mandilens` namespace and the configured log directory; it does not reconfigure the root Python logger.

In [7]:
from mandilens.utils.logger import get_logger

logger = get_logger("notebooks.00_environment_setup")
logger.info("MandiLens India environment setup started.")

print("Centralized logger initialization: PASS")

2026-11-15 11:38:43 | INFO | mandilens.notebooks.00_environment_setup | MandiLens India environment setup started.


Centralized logger initialization: PASS


## 10. Audit installed project dependencies

The dependency inventory is derived from the repository's `requirements.txt` groups. This cell checks installed distribution metadata; it does not install or upgrade packages. Core data-engineering packages are identified separately from application, modelling and development extras.

In [8]:
from importlib import metadata

PACKAGE_IMPORT_NAMES = {
    "numpy": "numpy",
    "pandas": "pandas",
    "polars": "polars",
    "pyarrow": "pyarrow",
    "duckdb": "duckdb",
    "fsspec": "fsspec",
    "psutil": "psutil",
    "huggingface-hub": "huggingface_hub",
    "requests": "requests",
    "tqdm": "tqdm",
    "python-dotenv": "dotenv",
    "PyYAML": "yaml",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "plotly": "plotly",
    "scipy": "scipy",
    "scikit-learn": "sklearn",
    "statsmodels": "statsmodels",
    "xgboost": "xgboost",
    "lightgbm": "lightgbm",
    "catboost": "catboost",
    "joblib": "joblib",
    "streamlit": "streamlit",
    "fastapi": "fastapi",
    "uvicorn": "uvicorn",
    "pydantic": "pydantic",
    "jupyterlab": "jupyterlab",
    "notebook": "notebook",
    "ipykernel": "ipykernel",
    "ipywidgets": "ipywidgets",
    "pytest": "pytest",
    "pytest-cov": "pytest-cov",
    "ruff": "ruff",
}

CORE_PACKAGES = {
    "numpy", "pandas", "polars", "pyarrow", "duckdb", "python-dotenv",
    "PyYAML", "matplotlib", "seaborn", "plotly", "huggingface-hub", "ipykernel",
}

DEPENDENCY_AUDIT = []
for distribution_name, import_name in PACKAGE_IMPORT_NAMES.items():
    try:
        installed_version = metadata.version(distribution_name)
        installed = True
    except metadata.PackageNotFoundError:
        installed_version = None
        installed = False

    DEPENDENCY_AUDIT.append({
        "distribution": distribution_name,
        "version": installed_version or "—",
        "status": "INSTALLED" if installed else "MISSING",
        "group": "CORE" if distribution_name in CORE_PACKAGES else "ADDITIONAL",
        "import_name": import_name,
    })

MISSING_CORE_PACKAGES = [
    item["distribution"]
    for item in DEPENDENCY_AUDIT
    if item["group"] == "CORE" and item["status"] == "MISSING"
]
MISSING_ADDITIONAL_PACKAGES = [
    item["distribution"]
    for item in DEPENDENCY_AUDIT
    if item["group"] == "ADDITIONAL" and item["status"] == "MISSING"
]
CORE_DEPENDENCIES_READY = not MISSING_CORE_PACKAGES

from IPython.display import HTML, display
from html import escape

rows = "".join(
    "<tr>"
    f"<td>{escape(item['distribution'])}</td>"
    f"<td>{escape(item['version'])}</td>"
    f"<td>{escape(item['group'])}</td>"
    f"<td>{escape(item['status'])}</td>"
    "</tr>"
    for item in DEPENDENCY_AUDIT
)
display(HTML(
    "<table><thead><tr><th>Package</th><th>Version</th><th>Group</th><th>Status</th></tr></thead>"
    f"<tbody>{rows}</tbody></table>"
))

print(f"Core dependency audit: {'PASS' if CORE_DEPENDENCIES_READY else 'INCOMPLETE'}")
if MISSING_CORE_PACKAGES:
    print("Missing core packages: " + ", ".join(MISSING_CORE_PACKAGES))
if MISSING_ADDITIONAL_PACKAGES:
    print("Missing additional packages: " + ", ".join(MISSING_ADDITIONAL_PACKAGES))

if MISSING_CORE_PACKAGES or MISSING_ADDITIONAL_PACKAGES:
    print("Install the repository requirements with: python -m pip install -r requirements.txt")

Package,Version,Group,Status
numpy,2.5.4,CORE,INSTALLED
pandas,3.1.1,CORE,INSTALLED
polars,1.44.2,CORE,INSTALLED
pyarrow,26.0.1,CORE,INSTALLED
duckdb,2.0.0,CORE,INSTALLED
fsspec,2026.9.0,ADDITIONAL,INSTALLED
psutil,7.2.2,ADDITIONAL,INSTALLED
huggingface-hub,1.33.0,CORE,INSTALLED
requests,2.34.2,ADDITIONAL,INSTALLED
tqdm,4.70.1,ADDITIONAL,INSTALLED


Core dependency audit: PASS


## 11. Run small in-memory data-engineering smoke tests

These tests verify basic interoperability only. They deliberately use three rows and an in-memory DuckDB connection; they do not touch the historical dataset.

In [9]:
SMOKE_TEST_RESULTS = {}

if CORE_DEPENDENCIES_READY:
    import duckdb
    import numpy as np
    import pandas as pd
    import polars as pl
    import pyarrow as pa
    import yaml

    sample = pd.DataFrame({
        "commodity": ["Wheat", "Rice", "Maize"],
        "modal_price": np.array([2450, 3100, 2200], dtype=np.int64),
    })

    polars_sample = pl.from_pandas(sample)
    arrow_sample = pa.Table.from_pandas(sample, preserve_index=False)

    with duckdb.connect(database=":memory:") as connection:
        duckdb_result = connection.execute(
            "SELECT 42 AS validation_value"
        ).fetchone()[0]

    SMOKE_TEST_RESULTS = {
        "pandas_dataframe": len(sample) == 3,
        "numpy_array": sample["modal_price"].to_numpy().shape == (3,),
        "polars_conversion": polars_sample.height == 3,
        "pyarrow_conversion": arrow_sample.num_rows == 3,
        "duckdb_query": duckdb_result == 42,
        "yaml_parser": isinstance(yaml.safe_load("status: ready"), dict),
    }
    SMOKE_TESTS_PASSED = all(SMOKE_TEST_RESULTS.values())

    for test_name, passed in SMOKE_TEST_RESULTS.items():
        print(f"{'PASS' if passed else 'FAIL'} — {test_name}")
    print(f"Data-stack smoke tests: {'PASS' if SMOKE_TESTS_PASSED else 'FAIL'}")
else:
    SMOKE_TESTS_PASSED = False
    SMOKE_TEST_RESULTS = {"skipped_missing_core_dependencies": False}
    print("Smoke tests skipped: install all missing core dependencies, then rerun this cell.")

PASS — pandas_dataframe
PASS — numpy_array
PASS — polars_conversion
PASS — pyarrow_conversion
PASS — duckdb_query
PASS — yaml_parser
Data-stack smoke tests: PASS


## 12. Verify visualization and notebook components

The visualization libraries are imported without generating arbitrary charts. The goal is to confirm that their interfaces are available for later analytical notebooks.

In [10]:
VISUALIZATION_READY = False
JUPYTER_READY = False

try:
    import matplotlib
    import matplotlib.pyplot as plt
    import plotly
    import plotly.graph_objects as go
    import seaborn as sns

    # Construct and close a minimal figure to test the Matplotlib backend.
    smoke_figure = plt.figure(figsize=(2, 1))
    plt.close(smoke_figure)
    plotly_figure = go.Figure()
    assert plotly_figure is not None
    assert sns is not None

    VISUALIZATION_READY = True
    print("Matplotlib, Seaborn and Plotly: PASS")
except Exception as error:
    print(f"Visualization stack: INCOMPLETE ({type(error).__name__})")

try:
    import ipykernel
    from IPython import get_ipython

    JUPYTER_READY = get_ipython() is not None
    print(f"ipykernel package: {metadata.version('ipykernel')}")
    print(f"Active IPython kernel: {'YES' if JUPYTER_READY else 'NO'}")
except Exception as error:
    print(f"Jupyter runtime check: INCOMPLETE ({type(error).__name__})")

Matplotlib, Seaborn and Plotly: PASS
ipykernel package: 7.4.0
Active IPython kernel: YES


## 13. Check package consistency

`pip check` reports incompatible or missing declared package requirements in the active environment. A non-zero result is recorded for follow-up; it does not change package versions automatically.

In [11]:
import subprocess

try:
    pip_check = subprocess.run(
        [sys.executable, "-m", "pip", "check"],
        capture_output=True,
        text=True,
        timeout=90,
        check=False,
    )
    PIP_CHECK_PASSED = pip_check.returncode == 0
    PIP_CHECK_SUMMARY = (pip_check.stdout or pip_check.stderr).strip()
    if PIP_CHECK_PASSED:
        print("pip check: PASS — no broken requirements detected")
    else:
        print("pip check: REVIEW REQUIRED")
        print(PIP_CHECK_SUMMARY[:2000] or "pip check returned a non-zero exit code.")
except (OSError, subprocess.TimeoutExpired) as error:
    PIP_CHECK_PASSED = False
    PIP_CHECK_SUMMARY = f"Unable to complete pip check: {type(error).__name__}"
    print(PIP_CHECK_SUMMARY)

pip check: PASS — no broken requirements detected


## 14. Save the environment audit report

The report stores interpreter and package metadata, configuration validation and smoke-test results. It does not store `.env` contents, credentials, absolute project paths or dataset records.

In [12]:
import json
from datetime import datetime, timezone

package_versions = {
    item["distribution"]: (None if item["version"] == "—" else item["version"])
    for item in DEPENDENCY_AUDIT
}

ENVIRONMENT_REPORT = {
    "project": {
        "name": PROJECT_NAME,
        "version": PROJECT_VERSION,
    },
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "runtime": {
        "python_version": platform.python_version(),
        "python_supported": PYTHON_SUPPORTED,
        "implementation": platform.python_implementation(),
        "operating_system": platform.system(),
        "architecture": platform.machine(),
        "jupyter_kernel_active": JUPYTER_READY,
    },
    "configuration": {
        "project_config_loaded": True,
        "required_sections_valid": True,
        "dataset_schema_audit_status": PROJECT_CONFIG.get("dataset", {}).get("schema_audit", {}).get("schema_verified", None),
        "dataset_row_count_verified": PROJECT_CONFIG.get("dataset", {}).get("schema_audit", {}).get("actual_row_count_verified", None),
    },
    "dependencies": {
        "core_ready": CORE_DEPENDENCIES_READY,
        "missing_core": MISSING_CORE_PACKAGES,
        "missing_additional": MISSING_ADDITIONAL_PACKAGES,
        "versions": package_versions,
    },
    "smoke_tests": SMOKE_TEST_RESULTS,
    "visualization_stack_ready": VISUALIZATION_READY,
    "pip_check_passed": PIP_CHECK_PASSED,
}

REPORT_PATH = PATHS.artifacts_dir / "metadata" / "environment_setup_report.json"
REPORT_PATH.write_text(
    json.dumps(ENVIRONMENT_REPORT, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)

print("Environment report saved: " + PATHS.relative_path(REPORT_PATH).as_posix())

Environment report saved: artifacts/metadata/environment_setup_report.json


## 15. Final environment readiness summary

A successful setup requires a supported interpreter, valid project configuration, all core dependencies, passing data-stack smoke tests, and an active Jupyter kernel. Additional modelling/application packages and `pip check` findings are visible in the audit report and should be resolved before their respective project phases.

In [13]:
ENVIRONMENT_READY = all((
    PYTHON_SUPPORTED,
    CORE_DEPENDENCIES_READY,
    SMOKE_TESTS_PASSED,
    VISUALIZATION_READY,
    JUPYTER_READY,
    PIP_CHECK_PASSED,
))

summary_items = {
    "Supported Python interpreter": PYTHON_SUPPORTED,
    "Project configuration": True,
    "Core dependencies": CORE_DEPENDENCIES_READY,
    "Data-engineering smoke tests": SMOKE_TESTS_PASSED,
    "Visualization stack": VISUALIZATION_READY,
    "Active Jupyter kernel": JUPYTER_READY,
    "pip dependency consistency": PIP_CHECK_PASSED,
}

for label, passed in summary_items.items():
    print(f"{'PASS' if passed else 'REVIEW'} — {label}")

print("\n" + ("ENVIRONMENT READY" if ENVIRONMENT_READY else "ENVIRONMENT SETUP INCOMPLETE"))
if not ENVIRONMENT_READY:
    print("Resolve the REVIEW items, restart the MandiLens kernel, and rerun the notebook.")
else:
    logger.info("MandiLens India environment setup completed successfully.")
    print("Next notebook: notebooks/01_data_source_audit.ipynb")

2026-11-15 11:38:48 | INFO | mandilens.notebooks.00_environment_setup | MandiLens India environment setup completed successfully.


PASS — Supported Python interpreter
PASS — Project configuration
PASS — Core dependencies
PASS — Data-engineering smoke tests
PASS — Visualization stack
PASS — Active Jupyter kernel
PASS — pip dependency consistency

ENVIRONMENT READY
Next notebook: notebooks/01_data_source_audit.ipynb


## 16. Expected outcome and hand-off

When all checks pass:

- The active kernel runs supported Python 3.12 or 3.13.
- `mandilens` imports from this repository.
- Project paths, configuration and logging initialize correctly.
- Required data-engineering and visualization packages are available.
- In-memory Pandas, NumPy, Polars, PyArrow and DuckDB checks pass.
- The environment report exists at `artifacts/metadata/environment_setup_report.json`.

### Important limitations

- This notebook does **not** confirm access to the Hugging Face dataset, the number of Parquet files, the dataset's actual row count, the data schema, or the completeness of the historical records.
- The approximately 75.96 million rows and 26 yearly Parquet files are publisher-reported metadata in the project's configuration. They remain unverified until the source-audit and inventory notebooks are completed.
- No raw data should be downloaded in this notebook.

### Next development file

`notebooks/01_data_source_audit.ipynb` — inspect the dataset card, repository/file inventory, available Parquet objects, metadata and access method before implementing the downloader.